In [23]:
#from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [24]:
PROJECT_DIR = Path.cwd().parent
output_file = PROJECT_DIR / "data_processed" / "flatfox_rentals_zentralschweiz.csv"
gpkg_path = PROJECT_DIR / "geodata" / "swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"

In [25]:
def merge_flatfox_csvs(project_dir):
    """Liest alle Erhebungs-CSVs ein, entfernt identische Duplikate und behält die neueste Zeile je Inserat-ID."""
    
    project_dir = Path(project_dir)
    source_files = sorted((project_dir / "flatfox_data").glob("*/listings_all_fields.csv"))
    output_file = (project_dir / "data_processed" / "flatfox_listings_merged.csv")

    if not source_files:
        raise FileNotFoundError("Keine Erhebungsdateien gefunden.")

    frames = []
    report = []

    for source in source_files:
        frame = pd.read_csv(
            source,
            encoding="utf-8-sig",
            dtype="string",
            keep_default_na=False,
            na_values=[""],
        )

        frame["pk"] = frame["pk"].str.strip()

        if (frame["pk"].isna() | frame["pk"].eq("")).any():
            raise ValueError(f"Leere Inserat-ID in {source}")

        rows_read = len(frame)
        frame = frame.drop_duplicates()

        if frame["pk"].duplicated().any():
            raise ValueError(
                f"Widersprüchliche Zeilen mit gleicher pk in {source}"
            )

        report.append({
            "source_run": source.parent.name,
            "rows_read": rows_read,
            "unique_ids": len(frame),
            "identical_duplicates": rows_read - len(frame),
        })

        frame["source_run"] = source.parent.name
        frame["source_file"] = source.relative_to(project_dir).as_posix()
        frame["__run_at"] = pd.to_datetime(
            source.parent.name,
            format="%Y%m%dT%H%M%S_%fZ",
            utc=True,
        )
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined.sort_values("__run_at", kind="stable")

    if combined.empty:
        raise ValueError("Die Erhebungsdateien enthalten keine Inserate.")

    # Sichtungen über alle vorhandenen Erhebungen bestimmen.
    grouped = combined.groupby("pk")

    combined["first_seen_at"] = grouped["__run_at"].transform("min")
    combined["last_seen_at"] = grouped["__run_at"].transform("max")
    combined["scrape_count"] = grouped["source_run"].transform("nunique")

    # Je ID die gesamte neueste Zeile übernehmen.
    merged = (
        combined.drop_duplicates("pk", keep="last")
        .drop(columns="__run_at")
        .sort_values("pk")
        .reset_index(drop=True)
    )

    for column in ["first_seen_at", "last_seen_at"]:
        merged[column] = merged[column].dt.strftime(
            "%Y-%m-%dT%H:%M:%S.%fZ"
        )

    # Fertige Datei schreiben und danach das bisherige Ergebnis ersetzen.
    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary_file = output_file.with_suffix(".csv.tmp")
    merged.to_csv(temporary_file, index=False, encoding="utf-8-sig")
    temporary_file.replace(output_file)

    report_df = pd.DataFrame(report)

    print(f"Erhebungsdateien: {len(source_files)}")
    print(f"Eingelesene Zeilen: {report_df['rows_read'].sum()}")
    print(f"Eindeutige Inserate: {len(merged)}")
    print(f"Gespeichert: {output_file}")

    return merged, report_df

In [27]:
def filter_flatfox_rentals(df, gpkg_path, output_file):
    """
    Ordnet Inseraten anhand ihrer Koordinaten Kanton und Gemeinde zu.
    Dokumentiert uneindeutige oder fehlende Zuordnungen und speichert
    die Inserate der Zentralschweiz als CSV.
    """
    gpkg_path = Path(gpkg_path)
    output_file = Path(output_file)

    if not gpkg_path.is_file():
        raise FileNotFoundError(
            f"GeoPackage-Datei nicht gefunden: {gpkg_path}"
        )

    df_rentals_geo = df.copy().reset_index(drop=True)
    layers = gpd.list_layers(gpkg_path)

    # Hilfsfunktion: Gewünschte Schweizer Gebietsflächen laden
    def load_areas(layer_keyword):
        layer_names = layers.loc[
            layers["name"].str.contains(
                layer_keyword, case=False, na=False
            ),
            "name",
        ].tolist()

        if len(layer_names) != 1:
            raise ValueError(
                f"Layer für '{layer_keyword}' nicht eindeutig gefunden: "
                f"{layers['name'].tolist()}"
            )

        areas = gpd.read_file(gpkg_path, layer=layer_names[0])

        areas = areas.rename(columns={
            column: column.upper()
            for column in areas.columns
            if column != areas.geometry.name
        })

        if areas.geometry.name != "geometry":
            areas = areas.rename_geometry("geometry")

        return areas.loc[areas["ICC"].eq("CH")].copy()

    # Inserat-Koordinaten vorbereiten
    longitude = pd.to_numeric(df_rentals_geo["longitude"], errors="coerce")
    latitude = pd.to_numeric(df_rentals_geo["latitude"], errors="coerce")

    valid_coordinates = (
        longitude.between(-180, 180)
        & latitude.between(-90, 90)
    ).fillna(False)

    points = gpd.GeoDataFrame(
        index=df_rentals_geo.index[valid_coordinates],
        geometry=gpd.points_from_xy(
            longitude.loc[valid_coordinates],
            latitude.loc[valid_coordinates],
        ),
        crs="EPSG:4326",
    )

    # Punkte eindeutig einem Gebiet zuordnen
    def assign_areas(areas, id_column):
        assigned = pd.Series(
            pd.NA, index=df_rentals_geo.index, dtype="object"
        )
        status = pd.Series(
            "koordinaten_ungueltig", index=df_rentals_geo.index
        )

        if valid_coordinates.any():
            matches = gpd.sjoin(
                points.to_crs(areas.crs),
                areas[[id_column, "geometry"]],
                how="left",
                predicate="intersects",
            )

            # Mehrere Geometrien derselben ID zählen als ein Gebiet.
            candidates = matches.groupby(level=0)[id_column].agg(
                lambda values: tuple(sorted(set(values.dropna())))
            )

            assigned = candidates.map(
                lambda values: values[0] if len(values) == 1 else pd.NA
            ).reindex(df_rentals_geo.index)

            status.loc[valid_coordinates] = "kein_treffer"

            ambiguous_indices = candidates[
                candidates.map(len).gt(1)
            ].index

            status.loc[ambiguous_indices] = "mehrdeutiger_treffer"
            status.loc[assigned.notna()] = "zugeordnet"

        return assigned, status

    # Kantone zuordnen
    cantons = load_areas("kantonsgebiet")

    canton_codes = dict(enumerate(
        [
            "ZH", "BE", "LU", "UR", "SZ", "OW", "NW", "GL", "ZG",
            "FR", "SO", "BS", "BL", "SH", "AR", "AI", "SG", "GR",
            "AG", "TG", "TI", "VD", "VS", "NE", "GE", "JU",
        ],
        start=1,
    ))

    cantons["kanton_code"] = pd.to_numeric(
        cantons["KANTONSNUMMER"], errors="coerce"
    ).map(canton_codes)

    if cantons["kanton_code"].isna().any():
        raise ValueError("Unbekannte Kantonsnummer in den Geodaten.")

    canton_ids, canton_status = assign_areas(
        cantons, "kanton_code"
    )

    df_rentals_geo["kanton_geo"] = canton_ids.astype("string")
    df_rentals_geo["kanton_geo_status"] = canton_status

    # Gemeinden zuordnen
    municipalities = load_areas("hoheitsgebiet")

    # Gemeindegebiete auswählen, keine Seegebiete oder Kommunanzen.
    # Je nach Export liegt OBJEKTART als Zahl oder Bezeichnung vor.
    object_type = municipalities["OBJEKTART"]

    is_municipality = (
        pd.to_numeric(object_type, errors="coerce").eq(0)
        | object_type.astype("string")
            .str.strip()
            .str.lower()
            .eq("gemeindegebiet")
    ).fillna(False)

    municipalities = municipalities.loc[is_municipality].copy()

    if municipalities.empty:
        raise ValueError(
            "Keine Schweizer Gemeindegebiete gefunden. "
            "Bitte die Werte der Spalte OBJEKTART prüfen."
        )

    municipalities["gemeinde_bfs"] = pd.to_numeric(
        municipalities["BFS_NUMMER"], errors="coerce"
    ).astype("Int64")

    if municipalities["gemeinde_bfs"].isna().any():
        raise ValueError("Fehlende BFS-Gemeindenummer in den Geodaten.")

    municipality_ids, municipality_status = assign_areas(
        municipalities, "gemeinde_bfs"
    )

    df_rentals_geo["gemeinde_bfs"] = municipality_ids.astype("Int64")
    df_rentals_geo["gemeinde_geo_status"] = municipality_status

    municipality_names = (
        municipalities.drop_duplicates("gemeinde_bfs")
        .set_index("gemeinde_bfs")["NAME"]
        .to_dict()
    )

    df_rentals_geo["gemeinde_name"] = (
        df_rentals_geo["gemeinde_bfs"]
        .map(municipality_names)
        .astype("string")
    )

    # Auf die Zentralschweiz eingrenzen und CSV speichern
    central_cantons = ["LU", "UR", "SZ", "OW", "NW", "ZG"]

    df_central = df_rentals_geo.loc[
        df_rentals_geo["kanton_geo"].isin(central_cantons)
    ].copy()

    output_file.parent.mkdir(parents=True, exist_ok=True)
    df_central.to_csv(
        output_file, index=False, encoding="utf-8-sig"
    )

    return df_rentals_geo, df_central

In [33]:
df_merged, merge_report = merge_flatfox_csvs(PROJECT_DIR)

# Filter der Rohdaten
mask = (df_merged["offer_type"].eq("RENT") & df_merged["object_category"].isin(["APARTMENT", "HOUSE"])).fillna(False)

df_filtered = df_merged.loc[mask]

# Kantone zuordnen und Zentralschweizer Inserate speichern
df_rentals_geo, df_central = filter_flatfox_rentals(df_filtered, gpkg_path, output_file,)

display(merge_report)

print(f"\nZusammengeführte Inserate: {len(df_merged)}")
print(f"Mietwohnungen und Häuser: {len(df_rentals_geo)}")
print(f"Davon Zentralschweiz: {len(df_central)}")
print(f"CSV gespeichert: {output_file}")

print("\nStatus der Kantonszuordnung:")
display(df_rentals_geo["kanton_geo_status"].value_counts())

print("\nInserate pro Zentralschweizer Kanton:")

central_cantons = ["LU", "UR", "SZ", "OW", "NW", "ZG"]

display(df_central)

Erhebungsdateien: 10
Eingelesene Zeilen: 25997
Eindeutige Inserate: 3418
Gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/flatfox_listings_merged.csv


,source_run,rows_read,unique_ids,identical_duplicates
0,20261018T100611_875376Z,284,284,0
1,20261018T103104_510890Z,1509,1509,0
2,20261018T121452_635701Z,3008,3008,0
3,20261018T151426_580437Z,3008,3008,0
4,20261018T191116_940418Z,3017,3017,0
5,20261019T151108_866230Z,3022,3022,0
6,20261020T191239_840560Z,3049,3049,0
7,20261021T192550_227446Z,3035,3035,0
8,20261022T153515_050865Z,3037,3037,0
9,20261023T122548_028947Z,3028,3028,0



Zusammengeführte Inserate: 3418
Mietwohnungen und Häuser: 1598
Davon Zentralschweiz: 889
CSV gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/flatfox_rentals_zentralschweiz.csv

Status der Kantonszuordnung:


kanton_geo_status
zugeordnet    1598
Name: count, dtype: int64


Inserate pro Zentralschweizer Kanton:


,pk,slug,url,short_url,submit_url,status,created,offer_type,object_category,object_type,...,capability_set_ids,attribute_reported_underbuildinglaws,first_seen_at,last_seen_at,scrape_count,kanton_geo,kanton_geo_status,gemeinde_bfs,gemeinde_geo_status,gemeinde_name
0,1112177,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112177/,/1112177/,/en/listing/1112177/submit/,act,2024-02-19T12:41:36.786020+01:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-18T10:06:11.875376Z,2026-10-23T12:25:48.028947Z,10,LU,zugeordnet,1059,zugeordnet,Kriens
1,1112194,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112194/,/1112194/,/en/listing/1112194/submit/,act,2024-02-19T12:41:44.416739+01:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-18T10:06:11.875376Z,2026-10-23T12:25:48.028947Z,10,LU,zugeordnet,1059,zugeordnet,Kriens
2,1112211,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112211/,/1112211/,/en/listing/1112211/submit/,act,2024-02-19T12:41:51.887002+01:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-18T10:06:11.875376Z,2026-10-23T12:25:48.028947Z,10,LU,zugeordnet,1059,zugeordnet,Kriens
3,1112230,langsagestrasse-21-6010-kriens,/en/flat/langsagestrasse-21-6010-kriens/1112230/,/1112230/,/en/listing/1112230/submit/,act,2024-02-19T12:41:59.876138+01:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-18T10:06:11.875376Z,2026-10-23T12:25:48.028947Z,10,LU,zugeordnet,1059,zugeordnet,Kriens
4,1116786,industriestrasse-70-6300-zug,/en/flat/industriestrasse-70-6300-zug/1116786/,/1116786/,/en/listing/1116786/submit/,act,2024-02-22T10:49:05.317415+01:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-18T10:31:04.510890Z,2026-10-23T12:25:48.028947Z,9,ZG,zugeordnet,1711,zugeordnet,Zug
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586,86476892,weinmarkt-11-6004-luzern,/en/flat/weinmarkt-11-6004-luzern/86476892/,/86476892/,/en/listing/86476892/submit/,act,2026-10-23T08:14:57.572426+02:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-23T12:25:48.028947Z,2026-10-23T12:25:48.028947Z,1,LU,zugeordnet,1061,zugeordnet,Luzern
1588,86477315,schwertstrasse-26-6300-zug,/en/flat/schwertstrasse-26-6300-zug/86477315/,/86477315/,/en/listing/86477315/submit/,act,2026-10-23T10:04:49.554908+02:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-23T12:25:48.028947Z,2026-10-23T12:25:48.028947Z,1,ZG,zugeordnet,1711,zugeordnet,Zug
1591,86477356,muhlemattstrasse-14-6004-luzern,/en/flat/muhlemattstrasse-14-6004-luzern/86477...,/86477356/,/en/listing/86477356/submit/,act,2026-10-23T10:09:43.805161+02:00,RENT,APARTMENT,APARTMENT,...,[],False,2026-10-23T12:25:48.028947Z,2026-10-23T12:25:48.028947Z,1,LU,zugeordnet,1061,zugeordnet,Luzern
1593,86477818,8808-pfaffikon-sz,/en/flat/8808-pfaffikon-sz/86477818/,/86477818/,/en/listing/86477818/submit/,act,2026-10-23T11:44:40.429591+02:00,RENT,APARTMENT,FURNISHED_FLAT,...,[],False,2026-10-23T12:25:48.028947Z,2026-10-23T12:25:48.028947Z,1,SZ,zugeordnet,1322,zugeordnet,Freienbach
